In [ ]:
# History tickets update plan

# alias: serve_tickets  →  serve_tickets_20261101-0300 (live)

# 1. Read all points (payload + vectors) from the live collection   ← already PII-masked
# 2. Fetch all SciLifeLab Serve incident ids from Edge              ← ids only, cheap
# 3. Diff = ticket_ids in Edge but not in the live collection
# 4. For the diff only: fetch the data card → keep only Closed/Resolved
#    → PII masking with our LLM → chunk → embed                     ← only step that touches raw data / calls the LLM
# 5. Create serve_tickets_20261215-0300 and write the old and new points into it
# 6. Check: point count = old count + new count; regex-scan for e-mail / phone
#    patterns to make sure no PII slipped through
# 7. If the checks pass, switch the alias to the new collection atomically;
#    keep the last 2–3 versions for rollback and delete older ones


In [1]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path

import requests
from dotenv import load_dotenv

# Load credentials from dev.env at the repo root (the notebook runs from notebooks/)
load_dotenv(Path.cwd().parent / "dev.env")

HOST = "https://edge.sys.kth.se"
SEARCH_URL = f"{HOST}/api/itsm/search.ws"
EDGE_API_USER = os.getenv("EDGE_API_USER")
EDGE_API_PASSWORD = os.getenv("EDGE_API_PASSWORD")
if not EDGE_API_USER or not EDGE_API_PASSWORD:
    raise RuntimeError("EDGE_API_USER / EDGE_API_PASSWORD not found in dev.env")
AUTH = (EDGE_API_USER, EDGE_API_PASSWORD)


def search(query, timeout=60):
    """Run one query against search.ws (Basic Auth, same as the working cell above) and return the XML root."""
    r = requests.get(SEARCH_URL, params={"query": query}, auth=AUTH, timeout=timeout)
    print("Status code:", r.status_code)
    print("Query:", query)
    print("URL:", r.url)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    # Edge returns query errors (e.g. syntax errors) with HTTP 200, so check the body too
    if root.tag == "error":
        raise RuntimeError(f"{root.findtext('code')}: {root.findtext('description')} | {root.findtext('details')}")
    return root

In [2]:
# 0) Connectivity check (no login needed)
# 1) No separate login/token step: every request sends Basic Auth directly
r = requests.get(f"{HOST}/echo", timeout=30)
print("Status code:", r.status_code)
print("URL:", r.url)

Status code: 200
URL: https://edge.sys.kth.se:443/idm/AuthnEngine


In [3]:
# 2) List all templates (shows every template code)
root = search("SELECT template FROM template")
template_codes = []
for template in root.findall("template"):
    code = template.findtext("code")
    template_codes.append(code)
print(len(template_codes), template_codes[:20])

Status code: 200
Query: SELECT template FROM template
URL: https://edge.sys.kth.se/api/itsm/search.ws?query=SELECT+template+FROM+template
2 ['person', 'incident']


In [4]:
# 3) Definition of one template (its attribute codes)
root = search("SELECT template FROM template WHERE template.code='incident'")
attribute_codes = []
for attribute in root.iter("attribute"):
    attribute_codes.append(attribute.findtext("code"))
print(attribute_codes)

Status code: 200
Query: SELECT template FROM template WHERE template.code='incident'
URL: https://edge.sys.kth.se/api/itsm/search.ws?query=SELECT+template+FROM+template+WHERE+template.code%3D%27incident%27
['send_kb', 'comment', 'e-mail', 'file_attachments', 'inc_reminders', 'created', 'updated', 'creator', 'latest_update_by', 'workflow_status', 'eup_latest_update', 'current_efecte_user', 'import_customer_information', 'monitor_worklog', 'support_group_e_mail_notices', 'support_person_email_notices', 'monitor_support_group', 'wf_status_solving', 'updated_by_mailtask', 'monitor_email', 'support_group_email', 'current_viewer', 'direct_url', 'from', 'reply_to', 'reassigned_check', 'is_new_datacard', 'find_computers', 'notes', 'internal_comments', 'remove_last_comment', 'time_report', 'task', 'add_time_spent', 'total_time_spent_old', 'time_log', 'remove_last_comment_exp', 'total_time_spent', 'latest_worklog', 'affiliation', 'school', 'department', 'unit_1', 'unit_2', 'study_program', 'c_la

In [5]:
# 4) Find real incident ids
# The query language has no LIMIT, so select only ids and narrow by an id range instead.
# Incident ids live in the 100000000-999999999 range (~10s, around 1000 ids).
root = search(
    "SELECT entity.id FROM entity WHERE entity.template.code='incident' "
    "AND entity.id > 100000000 AND entity.id < 999999999",
    timeout=120,
)
incident_ids = []
for element in root.findall("string"):
    incident_ids.append(element.text)
print(len(incident_ids), "incident ids, first 5:", incident_ids[:5])

Status code: 200
Query: SELECT entity.id FROM entity WHERE entity.template.code='incident' AND entity.id > 100000000 AND entity.id < 999999999
URL: https://edge.sys.kth.se/api/itsm/search.ws?query=SELECT+entity.id+FROM+entity+WHERE+entity.template.code%3D%27incident%27+AND+entity.id+%3E+100000000+AND+entity.id+%3C+999999999
1522 incident ids, first 5: ['193974479', '194097203', '307198528', '221488202', '264138935']


In [7]:
# 5) Fetch a single data card by a real id from step 4 (e.g. 193974479)
# Only whitelisted fields are printed: metadata plus the e-mail subject and body
# (subject / description; the entity name is the subject too). Everything else
# (e-mail addresses, person fields, comments, mail headers, ...) can hold PII,
# so it is shown as "<masked, N chars>" instead of its value.
SAFE_FIELDS = {
    "efecte_id", "status", "created", "updated", "modified", "created_-_customer", "resolved_-_customer",
    "contact_type", "request_type", "impact", "urgency", "priority", "deadline", "target_response_time",
    "initial_support_group", "support_group", "service", "initial_service", "category", "subcategory",
    "incident_opened", "incident_resolved", "incident_closed", "reopened", "first_pass_resolution",
    "response_time", "resolution_time", "total_process_time", "customer_resolution_time",
    "target_response_time_achieved", "target_resolution_time_achieved",
    "number_of_support_group_changes", "number_of_support_person_changes", "direct_url",
    "subject", "description",
}

incident_id = incident_ids[0]
root = search(f"SELECT entity FROM entity WHERE entity.id={incident_id}")
entity = root.find("entity")
print("Name:", entity.get("name"))
for attribute in entity.findall("attribute"):
    code = attribute.get("code")
    values = []
    for value in attribute.findall("value"):
        values.append(value.text)
    if code in SAFE_FIELDS:
        print(f"  {code}: {values[:3]}")
    elif values:
        n_chars = 0
        for value in values:
            n_chars += len(value or "")
        print(f"  {code}: <masked, {n_chars} chars>")

Status code: 200
Query: SELECT entity FROM entity WHERE entity.id=193974479
URL: https://edge.sys.kth.se/api/itsm/search.ws?query=SELECT+entity+FROM+entity+WHERE+entity.id%3D193974479
Name: KTH-INC-4472207-Failed to publish a shiny app on SciLifeLab Serve
  created: ['2023-11-27 09:37']
  updated: ['2023-12-18 22:53']
  creator: <masked, 8 chars>
  latest_update_by: <masked, 15 chars>
  support_group_e_mail_notices: <masked, 3 chars>
  support_person_email_notices: <masked, 3 chars>
  monitor_support_group: <masked, 32 chars>
  monitor_email: <masked, 32 chars>
  support_group_email: <masked, 19 chars>
  direct_url: ['https://edge.sys.kth.se/itsm/Frameset.do?a=workspace/datacard/view/193974479']
  from: <masked, 19 chars>
  total_time_spent_old: <masked, 1 chars>
  total_time_spent: <masked, 1 chars>
  second_email: <masked, 13 chars>
  efecte_id: ['KTH-INC-4472207']
  status: ['7 - Closed']
  contact_type: ['E-mail']
  subject: ['Failed to publish a shiny app on SciLifeLab Serve']
  d

In [ ]:
# 6) Find SciLifeLab Serve incidents newer than the last seen id
# support_group / service / category come back empty for this API user, so filter on
# initial_support_group (the group the ticket was first assigned to).
# Ids are increasing but not contiguous: "entity.id > last_seen_id" still catches every new ticket.
SUPPORT_GROUP = "SciLifeLab Serve"
METADATA_FIELDS = ["efecte_id", "created", "status", "initial_support_group"]
last_seen_id = 310000000  # in the poller, load this from storage instead

root = search(
    "SELECT entity.id FROM entity WHERE entity.template.code='incident' "
    f"AND $initial_support_group$ = '{SUPPORT_GROUP}' "
    f"AND entity.id > {last_seen_id}",
    timeout=120,
)
serve_ids = []
for element in root.findall("string"):
    serve_ids.append(int(element.text))
serve_ids.sort()
print(len(serve_ids), f"{SUPPORT_GROUP} incidents with id >", last_seen_id)

# Print only metadata (no subject, description or e-mail fields)
for serve_id in serve_ids:
    entity = search(f"SELECT entity FROM entity WHERE entity.id={serve_id}").find("entity")
    metadata = {"id": serve_id}
    for attribute in entity.findall("attribute"):
        code = attribute.get("code")
        if code in METADATA_FIELDS:
            metadata[code] = attribute.findtext("value")
    print(metadata)

if serve_ids:
    last_seen_id = serve_ids[-1]
print("new last_seen_id:", last_seen_id)